# Transaction-only baselines

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / "config.json").exists():
    ROOT = ROOT.parent
REPORTS = ROOT / "reports"
assert (REPORTS / "run_manifest.json").exists(), "Run python -m fraudgraph.pipeline --download first"
manifest = json.loads((REPORTS / "run_manifest.json").read_text())
print("Evidence run:", manifest["run_id"])
def figure(name):
    display(Image(filename=str(REPORTS / "figures" / name)))


Evidence run: 20260923T163139886696Z


The primary baseline uses 15 named intrinsic attributes. A sensitivity analysis adds 93 anonymized local features. Supplied neighborhood aggregates and graph degree columns are excluded from both baselines. Logistic Regression is standardized and class-balanced; XGBoost uses a fixed 250-tree specification.

In [2]:
from fraudgraph.data import LOCAL
display(pd.Series(LOCAL, name="Primary predictors"))
r = pd.read_csv(REPORTS / "results.csv")
display(r[r.experiment_id.str.endswith("_tx")][["experiment_id","split","pr_auc_ap","precision","recall","false_positive_rate","threshold"]])

0                total_BTC
1                     fees
2                     size
3      num_input_addresses
4     num_output_addresses
5               in_BTC_min
6               in_BTC_max
7              in_BTC_mean
8            in_BTC_median
9             in_BTC_total
10             out_BTC_min
11             out_BTC_max
12            out_BTC_mean
13          out_BTC_median
14           out_BTC_total
Name: Primary predictors, dtype: str

,experiment_id,split,pr_auc_ap,precision,recall,false_positive_rate,threshold
0,named_logistic_tx,validation,0.144363,0.205796,0.738921,0.371813,0.688297
1,named_logistic_tx,test,0.090574,0.084102,0.831761,0.546170,0.688297
4,named_xgboost_tx,validation,0.764254,0.869624,0.623314,0.012184,0.663754
5,named_xgboost_tx,test,0.423938,0.728000,0.429245,0.009670,0.663754
8,extended_logistic_tx,validation,0.346752,0.493442,0.761079,0.101872,0.891263
9,extended_logistic_tx,test,0.149566,0.202393,0.638365,0.151688,0.891263
12,extended_xgboost_tx,validation,0.959493,0.974725,0.854528,0.002889,0.720116
13,extended_xgboost_tx,test,0.634914,0.846547,0.520440,0.005688,0.720116


In [3]:
display(pd.read_csv(REPORTS / "splits.csv"))
display(Markdown((REPORTS / "LEAKAGE_AUDIT.md").read_text()))

,split,class,count
0,test,1,636
1,test,2,10548
2,test,3,35463
3,train,1,2871
4,train,2,23510
5,train,3,94423
6,validation,1,1038
7,validation,2,7961
8,validation,3,27319


# Leakage and prediction-time audit

## Prediction contract

The prediction unit is a Bitcoin transaction. The target is the dataset's known illicit label, not a proven legal conclusion or a chargeback outcome. Scores are computed at the **end of the transaction's time bucket**. All transactions and observed edges in that bucket are assumed available; labels in that bucket are unavailable. This is batch retrospective risk triage, not instantaneous authorization.

The dataset has 49 coarse time steps and no sufficient per-edge arrival timestamps for an honest within-step online replay. Out-degree, components, and PageRank can contain information that would arrive after the individual transaction but before bucket close. They pass the stated batch contract only. Do not describe the results as real-time detection.

## Split and fitting

- Train: steps 1â€“29. Validation: 30â€“39. Test: 40â€“49.
- Boundaries and the eight model/feature combinations were chosen before seeing model results.
- ID sets are disjoint. No random row split and no target encoding.
- Unknown class 3 maps to a missing outcome, never to legitimate class 0.
- Imputation and scaling fit on known-label training rows only, within sklearn pipelines.
- Validation chooses an F1 threshold per model and the operational model by average precision. Test results do not select hyperparameters, policy model, or thresholds.
- Both graph and transaction versions use identical splits, learner hyperparameters, label handling, and threshold-selection procedures.
- No early stopping or large search; 250 boosted trees and one logistic specification per feature set. Logistic uses balanced class weights; XGBoost uses unweighted log loss. Scores are not calibrated probabilities.

## Feature provenance

The primary baseline uses only 15 named intrinsic transaction attributes: value, fees, bytes, address counts, and distributions of input/output BTC amounts. The names support transaction-level interpretation, but the source's exact extraction code is not supplied here and cannot be independently reconstructed from anonymized IDs.

The primary baseline excludes the 93 anonymized local columns, 72 supplied aggregate columns, supplied in/out graph degrees, transaction ID, time-step index, and labels. A separately labeled sensitivity experiment adds the 93 local columns to both arms. Their source normalization and exact availability are not fully auditable; sensitivity results are not stronger production-validity evidence than the primary experiment.

Graph features are recomputed from visible nodes and edges for each cutoff, using no labels and no supplied aggregate features. Unknown nodes participate in topology because their edges are observable, not because their class is inferred. Each row is frozen at its own cutoff. There is no full-future graph computation followed by a split.

## Automated checks and evidence

`tests/test_graph.py` perturbs future nodes/edges and labels and verifies that past features remain unchanged. It checks degree orientation, components, isolates, and ID alignment. `tests/test_data.py` rejects duplicate IDs, dangling endpoints, and inconsistent embedded targets; it verifies unknown-label handling. `tests/test_evaluation.py` checks temporal boundaries, tie handling, review capacity, thresholds, and confusion counts.

`data_validation.json` records IDs, missing values, labels, time edges, file hashes, and schema checks. `duplicate_audit.json` records identical named-feature fingerprints shared across splits. Equal values on distinct IDs do not demonstrate duplicate transactions; they can represent ordinary repeated transaction patterns. All-missing attribute rows also share fingerprints. They remain in the primary population, with a separate robustness analysis excluding test fingerprints seen in training. Unknown outcome missingness is not assumed random.

## Residual risks

All observed edges are within a time bucket. Thus the holdout tests later disconnected graph populations, not connected historical propagation. Entity-level duplication cannot be ruled out without usable wallet identity. Labels may have been assigned using evidence obtained after the nominal time step; label availability dates are absent. The experiment assumes training outcomes are mature by fitting time. Dataset construction and selective labeling can inflate apparent performance relative to deployment.

There are only ten test time buckets. A paired time-bucket bootstrap is a descriptive uncertainty estimate, not proof across independent markets. Temporal dependence between buckets and large regime shifts limit its interpretation. Inspect `test_by_time.csv`, rather than relying on pooled AP alone. The unknown population prevents identification of true population precision, recall, or false-positive rates. Capacity reports show known outcomes and precision bounds on all traffic.


Average precision is the non-interpolated PR summary used throughout. Threshold metrics use the validation-F1 optimum frozen before test evaluation. Accuracy is intentionally not the model-selection objective.